## Indexing

### kid-friend

In [1]:
import json
import pandas as pd


lines = []
with open(r'../data/kid-friend-en/en/inputs/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
kid_friend_df_final = pd.DataFrame(line_dicts)
kid_friend_df_final.drop_duplicates(inplace=True,ignore_index=True)
kid_friend_df_final.head(1)

,docno,snippet,title,main_content
0,6e421f1539b1457b853712d81be87743,WEBBTS (also Bangtan Boys; Korean: 방탄소년단 Bangt...,BTS (band) - Wikipedia,BTS (also Bangtan Boys; Korean: 방탄소년단 Bangtan ...


In [2]:
print("Number of documents: ", len(kid_friend_df_final))
print("Longest document length (in characters): ", max([len(text) for text in kid_friend_df_final["main_content"]]))
print("Longest length of document title (in characters): ", max([len(title) for title in kid_friend_df_final["title"]]))
print("Maximum length of doc id (in characters): ", max([len(docno) for docno in kid_friend_df_final["docno"]]))

Number of documents:  2385
Longest document length (in characters):  117554
Longest length of document title (in characters):  452
Maximum length of doc id (in characters):  32


In [4]:
from pathlib import Path
import pyterrier as pt
import pandas as pd

kid_friend_df_final.to_dict(orient="records")

kid_friend_idx_path = "../data/kid-friend-en/index"
kid_friend_indexer = pt.IterDictIndexer(
    index_path = str(kid_friend_idx_path),
    meta={ # metadata recorded in index
        "docno": max([len(docno) for docno in kid_friend_df_final["docno"]]),
        "title": max([len(title) for title in kid_friend_df_final["title"]]),
        "snippet": max([len(snippet) for snippet in kid_friend_df_final["snippet"]]),
        "text": max([len(main_content) for main_content in kid_friend_df_final["main_content"]])
    },
    text_attrs = ["text"], # columns indexed
    stemmer="porter",
    stopwords="terrier",
)

Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [5]:
def kid_friend_iter():
    for i, row in kid_friend_df_final.iterrows():
        yield {
            "docno": row["docno"],
            "title": row["title"],
            "snippet": row["snippet"],
            "text": row["main_content"]
        }

In [6]:
kid_friend_index_ref = kid_friend_indexer.index(kid_friend_iter())

11:09:49.545 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (37d53c33110a4231807e71e377f08377) - further warnings are suppressed
11:09:53.366 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 187 empty documents


In [7]:
len(kid_friend_df_final.loc[kid_friend_df_final["main_content"]==""])

187

### requik

In [9]:
lines = []
with open(r'../data/requik/corpus/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
requik_df_final = pd.DataFrame(line_dicts)
requik_df_final = requik_df_final.rename({"id":"docno", "contents": "main_content"}, axis=1)
requik_df_final.drop_duplicates(inplace=True,ignore_index=True)
requik_df_final.head(1)

,docno,main_content
0,RQ_d1,When was the last time the capulin volcano nat...


In [10]:
print("Number of documents: ", len(requik_df_final))
print("Longest document length (in characters): ", max([len(text) for text in requik_df_final["main_content"]]))
print("Maximum length of doc id (in characters): ", max([len(docno) for docno in requik_df_final["docno"]]))

Number of documents:  317
Longest document length (in characters):  1281
Maximum length of doc id (in characters):  7


In [13]:
requik_df_final.to_dict(orient="records")

requik_idx_path = "../data/requik/index"
requik_indexer = pt.IterDictIndexer(
    index_path = str(requik_idx_path),
    meta={ # metadata recorded in index
        "docno": max([len(docno) for docno in requik_df_final["docno"]]),
        "text": max([len(main_content) for main_content in requik_df_final["main_content"]])
    },
    text_attrs = ["text"], # columns indexed
    stemmer="porter",
    stopwords="terrier",
)

In [14]:
def requik_iter():
    for i, row in requik_df_final.iterrows():
        yield {
            "docno": row["docno"],
            "text": row["main_content"]
        }

In [15]:
requik_index_ref = requik_indexer.index(requik_iter())

In [17]:
len(requik_df_final.loc[requik_df_final["main_content"]==""])

0

## Retrieval

In [1]:
from pathlib import Path
import pyterrier as pt
import pandas as pd

kid_friend_queries = pd.read_csv("../data/kid-friend-en/en/inputs/topics.csv")
kid_friend_queries["qid"] = kid_friend_queries["qid"].astype(str)
# kid_friend_topical_qrels = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
# kid_friend_topical_qrels["qid"] = kid_friend_topical_qrels["qid"].astype(str)

kid_friend_idx_path = "../data/kid-friend-en/index"
kid_friend_index = pt.IndexFactory.of(str(kid_friend_idx_path))

Java started (triggered by IndexFactory.of) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [2]:
requik_queries = pd.read_csv("../data/requik/inputs/topics.csv")
requik_queries["qid"] = requik_queries["qid"].astype(str)

requik_idx_path = "../data/requik/index"
requik_index = pt.IndexFactory.of(str(requik_idx_path))

In [3]:
from pyterrier_caching import RetrieverCache

## simple first stage retrievers
kid_friend_tf_idf = pt.terrier.Retriever(kid_friend_index, wmodel="TF_IDF")
kid_friend_bm25 = pt.terrier.Retriever(kid_friend_index, wmodel="BM25")
kid_friend_dirichlet_lm = pt.terrier.Retriever(kid_friend_index, wmodel="DirichletLM")

requik_tf_idf = pt.terrier.Retriever(requik_index, wmodel="TF_IDF")
requik_bm25 = pt.terrier.Retriever(requik_index, wmodel="BM25")
requik_dirichlet_lm = pt.terrier.Retriever(requik_index, wmodel="DirichletLM")

# caching the first 100 resources retrieved by bm25 for each dataset to be used for re-rankers
kid_friend_bm25_cached_retriever = RetrieverCache('./cache/kid-friend-en', kid_friend_bm25 % 100 >> pt.text.get_text(kid_friend_index, 'text'))
requik_bm25_cached_retriever = RetrieverCache('./cache/requik', requik_bm25 % 100 >> pt.text.get_text(requik_index, 'text'))

In [17]:
type(kid_friend_tf_idf)

pyterrier.terrier.retriever.Retriever

In [4]:
## neural re-rankers: monoT5

from pyterrier_t5 import MonoT5ReRanker

monoT5 = MonoT5ReRanker(text_field='text', model = 'castorini/monot5-base-msmarco')
kid_friend_monoT5 = (kid_friend_bm25_cached_retriever >> monoT5)
requik_monoT5 = (requik_bm25_cached_retriever >> monoT5)

/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-04 13:11:20.923242: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-04 13:11:20.974766: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-04 13:11:22.134478: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You

In [13]:
# kid_friend_bm25.search(kid_friend_queries["query"][2])

,qid,docid,docno,rank,score,query
0,1,554,1034d422a9154fd09d1ef847809c4d88,0,24.403817,BTS merch
1,1,565,42f8dede7c1a4818a7d91ac2b4486094,1,23.978815,BTS merch
2,1,562,6a7d04e19d5c4c589749780c39dab51f,2,23.971948,BTS merch
3,1,29,f449e7fd662e4a5f8f46fe67a20f2a85,3,23.709059,BTS merch
4,1,23,63175e0a62614f568ba8bb88537dfd97,4,22.455583,BTS merch
...,...,...,...,...,...,...
85,1,1379,efba881a2b0b4364852252b4144ca8f5,85,5.963200,BTS merch
86,1,703,7383555fdeb740df9f7c61f76b6fc73f,86,4.722670,BTS merch
87,1,759,6d39234cd20d46989480ebbbfe0407b1,87,2.169711,BTS merch
88,1,683,9ffc2553678d4372a329524ea974eb6d,88,2.077047,BTS merch


In [14]:
# kid_friend_monoT5.search(kid_friend_queries["query"][2])

monoT5: 100%|██████████| 23/23 [00:02<00:00, 11.11batches/s]


,qid,docid,docno,query,text,score,rank
16,1,1725,c51ba51158df477b8616c7bda055788e,BTS merch,BTS - Seoul-Mate\n\nBTS\n\nFilter\n 143 pro...,-0.001540,0
11,1,1728,ec742a5f53bf4c6db7aafdc361afbf61,BTS merch,Skip to content\n10% new customer discount - C...,-0.001887,1
15,1,24,06c8038c50714812ae1fb562865444a1,BTS merch,Skip to content\n10% new customer discount - C...,-0.002117,2
3,1,29,f449e7fd662e4a5f8f46fe67a20f2a85,BTS merch,BTS Merch Germany : Clothing and accessories\n...,-0.002126,3
0,1,554,1034d422a9154fd09d1ef847809c4d88,BTS merch,Previous\nNext\n\nBTS merch categories\n\nBTS ...,-0.004141,4
...,...,...,...,...,...,...,...
89,1,353,9d5f4572038d4855bc2e53f2b799b833,BTS merch,This article also exists as an audio file.\n\n...,-12.699072,85
56,1,1376,9cf39de35fe944e2b83d91ad03a30cf6,BTS merch,Skip to content\n\nBangkok/Siam\n\nUseful arti...,-12.955190,86
87,1,759,6d39234cd20d46989480ebbbfe0407b1,BTS merch,Jump to content\n\nGladiator II\n\nFrom Wikipe...,-13.358723,87
86,1,703,7383555fdeb740df9f7c61f76b6fc73f,BTS merch,List of Official Disney Princesses (And the On...,-13.463104,88


In [5]:
from pathlib import Path
from pyterrier.measures import RR, nDCG

kf_topical_qrels = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
kf_topical_qrels["qid"] = kf_topical_qrels["qid"].astype(str)

kf_results_dir = Path.cwd() / "results" / "kid-friend"
kf_results_dir.mkdir(exist_ok=True, parents=True)

pt.Experiment(
    retr_systems = [kid_friend_tf_idf, kid_friend_bm25, kid_friend_dirichlet_lm, kid_friend_monoT5],
    topics = kid_friend_queries,
    qrels = kf_topical_qrels,
    names=["TF-IDF", "BM25", "DirichletLM", "MonoT5-base"],
    eval_metrics=[RR @ 10, nDCG @ 10],
    save_dir=str(kf_results_dir),
    save_mode='overwrite' # switch to overwrite if required
)

requik_topical_qrels = pd.read_csv("../data/requik/qrels/qrels-relevance-binary.csv")
requik_topical_qrels["qid"] = requik_topical_qrels["qid"].astype(str)

requik_results_dir = Path.cwd() / "results" / "requik"
requik_results_dir.mkdir(exist_ok=True, parents=True)

pt.Experiment(
    retr_systems = [requik_tf_idf, requik_bm25, requik_dirichlet_lm, requik_monoT5],
    topics = requik_queries,
    qrels = requik_topical_qrels,
    names=["TF-IDF", "BM25", "DirichletLM", "MonoT5-base"],
    eval_metrics=[RR @ 10, nDCG @ 10],
    save_dir=str(requik_results_dir),
    save_mode='overwrite' # switch to overwrite if required
)

/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/pyterrier/pipelines.py:315: UserWarning: save_dir is set, but the file '/data/hc_storage/ChildCenteredEvaluation/code/results/kid-friend/TF-IDF.res.gz' already exists. If you are aware of are happy to reuse this file to speed up evaluation, set save_mode='reuse'; if you want to overwrite it, set save_mode='overwrite'. To make this condition an error, use save_mode='error'.
  warn(("save_dir is set, but the file '%s' already exists. If you are aware of are happy to reuse this " % save_file)+
/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/pyterrier/pipelines.py:315: UserWarning: save_dir is set, but the file '/data/hc_storage/ChildCenteredEvaluation/code/results/kid-friend/BM25.res.gz' already exists. If you are aware of are happy to reuse this file to speed up evaluation, set save_mode='reuse'; if you want to overwrite it, set save_mode='overwrite'. To make this condition an error, use save_mode='

,name,RR@10,nDCG@10
0,TF-IDF,0.743857,0.587289
1,BM25,0.763381,0.599215
2,DirichletLM,0.802889,0.583483
3,MonoT5-base,0.748167,0.643369


In [4]:
from rerank import LLMReRanker
### Note to self: run export 'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True' for each model first in a terminal, 
###               reset the kernel, and then run the model

# listwise rerankers
rank_vicuna = LLMReRanker("castorini/rank_vicuna_7b_v1")
kid_friend_vicuna = kid_friend_bm25_cached_retriever >> rank_vicuna
requik_vicuna = requik_bm25_cached_retriever >> rank_vicuna

/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-04 13:36:47.131599: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-04 13:36:47.182735: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-04 13:36:48.218602: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You

In [5]:
# kid_friend_zephyr.search(kid_friend_queries["query"][2])

In [6]:
from pathlib import Path
from pyterrier.measures import RR, nDCG

kf_topical_qrels = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
kf_topical_qrels["qid"] = kf_topical_qrels["qid"].astype(str)

kf_results_dir = Path.cwd() / "results" / "kid-friend"
kf_results_dir.mkdir(exist_ok=True, parents=True)

pt.Experiment(
    retr_systems = [kid_friend_bm25, kid_friend_vicuna],
    topics = kid_friend_queries,
    qrels = kf_topical_qrels,
    names=["BM25", "RankVicuna"],
    eval_metrics=[RR @ 10, nDCG @ 10],
    save_dir=str(kf_results_dir),
    save_mode='overwrite' # switch to overwrite if required
)

requik_topical_qrels = pd.read_csv("../data/requik/qrels/qrels-relevance-binary.csv")
requik_topical_qrels["qid"] = requik_topical_qrels["qid"].astype(str)

requik_results_dir = Path.cwd() / "results" / "requik"
requik_results_dir.mkdir(exist_ok=True, parents=True)

pt.Experiment(
    retr_systems = [requik_bm25, requik_vicuna],
    topics = requik_queries,
    qrels = requik_topical_qrels,
    names=["BM25", "RankVicuna"],
    eval_metrics=[RR @ 10, nDCG @ 10],
    save_dir=str(requik_results_dir),
    save_mode='overwrite' # switch to overwrite if required
)

/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/pyterrier/pipelines.py:315: UserWarning: save_dir is set, but the file '/data/hc_storage/ChildCenteredEvaluation/code/results/kid-friend/BM25.res.gz' already exists. If you are aware of are happy to reuse this file to speed up evaluation, set save_mode='reuse'; if you want to overwrite it, set save_mode='overwrite'. To make this condition an error, use save_mode='error'.
  warn(("save_dir is set, but the file '%s' already exists. If you are aware of are happy to reuse this " % save_file)+
  0%|          | 0/1 [00:00<?, ?window/s]Token indices sequence length is longer than the specified maximum sequence length for this model (5755 > 4096). Running this sequence through the model will result in indexing errors
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (4096). Depending on the model, you may observe exceptions, performance degradation, or nothing at

,name,RR@10,nDCG@10
0,BM25,0.277279,0.329358
1,RankVicuna,0.215239,0.277067


In [4]:
from rerank import LLMReRanker

rank_zephyr = LLMReRanker("castorini/rank_zephyr_7b_v1_full")
kid_friend_zephyr = kid_friend_bm25_cached_retriever >> rank_zephyr
requik_zephyr = requik_bm25_cached_retriever >> rank_zephyr

/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-04 14:56:13.700842: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-04 14:56:13.751763: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-04 14:56:14.790329: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You

In [6]:
from pathlib import Path
from pyterrier.measures import RR, nDCG

kf_topical_qrels = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
kf_topical_qrels["qid"] = kf_topical_qrels["qid"].astype(str)

kf_results_dir = Path.cwd() / "results" / "kid-friend"
kf_results_dir.mkdir(exist_ok=True, parents=True)

pt.Experiment(
    retr_systems = [kid_friend_bm25, kid_friend_zephyr],
    topics = kid_friend_queries,
    qrels = kf_topical_qrels,
    names=["BM25", "RankZephyr"],
    eval_metrics=[RR @ 10, nDCG @ 10],
    save_dir=str(kf_results_dir),
    save_mode='overwrite' # switch to overwrite if required
)

requik_topical_qrels = pd.read_csv("../data/requik/qrels/qrels-relevance-binary.csv")
requik_topical_qrels["qid"] = requik_topical_qrels["qid"].astype(str)

requik_results_dir = Path.cwd() / "results" / "requik"
requik_results_dir.mkdir(exist_ok=True, parents=True)

pt.Experiment(
    retr_systems = [requik_bm25, requik_zephyr],
    topics = requik_queries,
    qrels = requik_topical_qrels,
    names=["BM25", "RankZephyr"],
    eval_metrics=[RR @ 10, nDCG @ 10],
    save_dir=str(requik_results_dir),
    save_mode='overwrite' # switch to overwrite if required
)

/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/pyterrier/pipelines.py:315: UserWarning: save_dir is set, but the file '/data/hc_storage/ChildCenteredEvaluation/code/results/kid-friend/BM25.res.gz' already exists. If you are aware of are happy to reuse this file to speed up evaluation, set save_mode='reuse'; if you want to overwrite it, set save_mode='overwrite'. To make this condition an error, use save_mode='error'.
  warn(("save_dir is set, but the file '%s' already exists. If you are aware of are happy to reuse this " % save_file)+
100%|██████████| 1/1 [00:54<00:00, 54.87s/window]
/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/pyterrier/pipelines.py:315: UserWarning: save_dir is set, but the file '/data/hc_storage/ChildCenteredEvaluation/code/results/requik/BM25.res.gz' already exists. If you are aware of are happy to reuse this file to speed up evaluation, set save_mode='reuse'; if you want to overwrite it, set save_mode='overwrite'. To m

,name,RR@10,nDCG@10
0,BM25,0.277279,0.329358
1,RankZephyr,0.208416,0.288493


## Evaluation

In [5]:
import pyterrier as pt
from pathlib import Path
from pyterrier.measures import RR, nDCG
from utils import get_ranked_lists
import pandas as pd


kf_queries = pd.read_csv("../data/kid-friend-en/en/inputs/topics.csv")
kf_queries["qid"] = kf_queries["qid"].astype(str)

kf_results_dir = Path.cwd() / "results" / "kid-friend"
kf_results_dir.mkdir(exist_ok=True, parents=True)

qrels = [
    "relevance", 
    "readability", 
    "objectivity", 
    "edu"
]

for qrel in qrels:
    kf_qrel_file = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-"+qrel+"-binary.csv")
    kf_qrel_file["qid"] = kf_qrel_file["qid"].astype(str)

    if qrel == "relevance":
        metrics = [RR @ 10, nDCG @ 10]
    else:
        metrics = [RR@10]
    
    kf_results = pt.Experiment(
        retr_systems = get_ranked_lists("results/kid-friend/"),
        topics = kf_queries,
        qrels = kf_qrel_file,
        names=["TF-IDF", "BM25", "DirichletLM", "MonoT5", "RankVicuna", "RankZephyr"],
        eval_metrics=metrics,
        save_dir=str(kf_results_dir),
        save_mode='reuse', # switch to overwrite if required,
        # baseline=0,
        # correction='bonferroni'
    )
    
    kf_results.to_csv("results/kid-friend/"+qrel+"_eval.csv", index=False)
    print(qrel, "\n==============================================\n", kf_results, "\n==============================================\n\n")

relevance 
           name     RR@10   nDCG@10
0       TF-IDF  0.743857  0.587289
1         BM25  0.763381  0.599215
2  DirichletLM  0.802889  0.583483
3       MonoT5  0.748167  0.643369
4   RankVicuna  0.779381  0.602107
5   RankZephyr  0.808667  0.643584 


readability 
           name     RR@10
0       TF-IDF  0.469190
1         BM25  0.493048
2  DirichletLM  0.476556
3       MonoT5  0.496278
4   RankVicuna  0.509048
5   RankZephyr  0.498802 


objectivity 
           name     RR@10
0       TF-IDF  0.670000
1         BM25  0.671190
2  DirichletLM  0.678833
3       MonoT5  0.644333
4   RankVicuna  0.671190
5   RankZephyr  0.661667 


edu 
           name     RR@10
0       TF-IDF  0.198889
1         BM25  0.198889
2  DirichletLM  0.200333
3       MonoT5  0.162111
4   RankVicuna  0.198889
5   RankZephyr  0.176024 




In [6]:
import pyterrier as pt
from pathlib import Path
from pyterrier.measures import RR, nDCG
from utils import get_ranked_lists
import pandas as pd

requik_queries = pd.read_csv("../data/requik/inputs/topics.csv")
requik_queries["qid"] = requik_queries["qid"].astype(str)

requik_results_dir = Path.cwd() / "results" / "requik"
requik_results_dir.mkdir(exist_ok=True, parents=True)

qrels = [
    "relevance", 
    "readability", 
    "objectivity", 
    "edu"
]

for qrel in qrels:

    requik_qrel_file = pd.read_csv("../data/requik/qrels/qrels-"+qrel+"-binary.csv")
    requik_qrel_file["qid"] = requik_qrel_file["qid"].astype(str)

    if qrel == "relevance":
        metrics = [RR @ 10, nDCG @ 10]
    else:
        metrics = [RR@10]
    
    requik_results = pt.Experiment(
        retr_systems = get_ranked_lists("results/requik/"),
        topics = requik_queries,
        qrels = requik_qrel_file,
        names=["TF-IDF", "BM25", "DirichletLM", "MonoT5", "RankVicuna", "RankZephyr"],
        eval_metrics= metrics,
        save_dir=str(requik_results_dir),
        save_mode='reuse', # switch to overwrite if required
        # baseline=0,
        # correction='bonferroni'
    )
    
    requik_results.to_csv("results/requik/"+qrel+"_eval.csv", index=False)
    print(qrel, "\n==============================================\n", requik_results, "\n==============================================\n\n")    

relevance 
           name     RR@10   nDCG@10
0       TF-IDF  0.277449  0.329534
1         BM25  0.277279  0.329358
2  DirichletLM  0.296283  0.359240
3       MonoT5  0.277671  0.341833
4   RankVicuna  0.215239  0.277067
5   RankZephyr  0.208416  0.288493 


readability 
           name     RR@10
0       TF-IDF  0.214136
1         BM25  0.213966
2  DirichletLM  0.234142
3       MonoT5  0.235497
4   RankVicuna  0.188782
5   RankZephyr  0.177996 


objectivity 
           name     RR@10
0       TF-IDF  0.185342
1         BM25  0.185172
2  DirichletLM  0.207515
3       MonoT5  0.163187
4   RankVicuna  0.129452
5   RankZephyr  0.114107 


edu 
           name     RR@10
0       TF-IDF  0.002033
1         BM25  0.002033
2  DirichletLM  0.001742
3       MonoT5  0.002962
4   RankVicuna  0.003659
5   RankZephyr  0.003252 




In [1]:
import pyterrier as pt
from utils import RBP, get_ranked_lists
import pandas as pd


requik_retriever_systems = get_ranked_lists("results/requik/")
requik_retriever_names = ["TF-IDF", "BM25", "DirichletLM", "MonoT5", "RankVicuna", "RankZephyr"]

# requik_retriever_systems = [requik_tf_idf]
# requik_retriever_names = ["TF-IDF"]

requik_queries = pd.read_csv("../data/requik/inputs/topics.csv")
requik_queries["qid"] = requik_queries["qid"].astype(str)
requik_qrel = pd.read_csv("../data/requik/qrels/qrels-relevance-binary.csv")
requik_qrel["qid"] = requik_qrel["qid"].astype(str)

requik_rbp_results = []
for sys, name in zip(requik_retriever_systems, requik_retriever_names):
    score = RBP(topics=requik_queries, qrels=requik_qrel, retriever_system=sys, phi=0.8, perquery=False, k=10)
    requik_rbp_results.append([name, score])

requik_rbp_results_df = pd.DataFrame(requik_rbp_results, columns=["name", f"RBP@10"])
requik_rbp_results_df

/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-05 16:31:51.847613: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-05 16:31:51.900402: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-05 16:31:53.033294: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You

,name,RBP@10
0,TF-IDF,0.072359
1,BM25,0.072257
2,DirichletLM,0.078922
3,MonoT5,0.069298
4,RankVicuna,0.060186
5,RankZephyr,0.057536


In [2]:
import pyterrier as pt
from utils import cRBP, get_ranked_lists
import pandas as pd


requik_retriever_systems = get_ranked_lists("results/requik/")
requik_retriever_names = ["TF-IDF", "BM25", "DirichletLM", "MonoT5", "RankVicuna", "RankZephyr"]

# requik_retriever_systems = [requik_tf_idf]
# requik_retriever_names = ["TF-IDF"]

requik_queries = pd.read_csv("../data/requik/inputs/topics.csv")
requik_queries["qid"] = requik_queries["qid"].astype(str)
requik_qrel = pd.read_csv("../data/requik/qrels/qrels-relevance-binary.csv")
requik_qrel["qid"] = requik_qrel["qid"].astype(str)
requik_corpus = pd.read_csv("../data/requik/inputs/documents.csv")

requik_crbp_results = []
for sys, name in zip(requik_retriever_systems, requik_retriever_names):
    score = cRBP(topics=requik_queries, qrels=requik_qrel, corpus = requik_corpus, upper_threshold = 5, lower_threshold = 3, 
                 retriever_system=sys, phi=0.8, perquery=False, k=10)
    requik_crbp_results.append([name, score])

requik_crbp_results_df = pd.DataFrame(requik_crbp_results, columns=["name", f"cRBP@10"])
requik_crbp_results_df

,name,cRBP@10
0,TF-IDF,0.008077
1,BM25,0.008077
2,DirichletLM,0.008876
3,MonoT5,0.008125
4,RankVicuna,0.007378
5,RankZephyr,0.007325


In [8]:
requik_rbp_results = requik_rbp_results_df.merge(requik_crbp_results_df, on="name")
requik_rbp_results.to_csv("results/requik/rbp_eval.csv", index=False)
requik_rbp_results

,name,RBP@10,cRBP@10
0,TF-IDF,0.072359,0.008077
1,BM25,0.072257,0.008077
2,DirichletLM,0.078922,0.008876
3,MonoT5,0.069298,0.008125
4,RankVicuna,0.060186,0.007378
5,RankZephyr,0.057536,0.007325


In [1]:
import pyterrier as pt
from utils import RBP, get_ranked_lists
import pandas as pd


kf_retriever_systems = get_ranked_lists("results/kid-friend/")
kf_retriever_names = ["TF-IDF", "BM25", "DirichletLM", "MonoT5", "RankVicuna", "RankZephyr"]

# kf_retriever_systems = [kid_friend_tf_idf]
# kf_retriever_names = ["TF-IDF"]

kf_queries = pd.read_csv("../data/kid-friend-en/en/inputs/topics.csv")
kf_queries["qid"] = kf_queries["qid"].astype(str)
kf_qrel_file = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
kf_qrel_file["qid"] = kf_qrel_file["qid"].astype(str)

kf_rbp_results = []
for sys, name in zip(kf_retriever_systems, kf_retriever_names):
    score = RBP(topics=kf_queries, qrels=kf_qrel_file, retriever_system=sys, phi=0.8, perquery=False, k=10)
    kf_rbp_results.append([name, score])

kf_rbp_results_df = pd.DataFrame(kf_rbp_results, columns=["name", f"RBP@10"])
kf_rbp_results_df

/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-05 16:47:53.718232: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-05 16:47:53.769245: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-05 16:47:54.886354: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You

,name,RBP@10
0,TF-IDF,0.536844
1,BM25,0.540305
2,DirichletLM,0.527164
3,MonoT5,0.300691
4,RankVicuna,0.543076
5,RankZephyr,0.309961


In [2]:
import pyterrier as pt
from utils import cRBP, get_ranked_lists
import pandas as pd


kf_retriever_systems = get_ranked_lists("results/kid-friend/")
kf_retriever_names = ["TF-IDF", "BM25", "DirichletLM", "MonoT5", "RankVicuna", "RankZephyr"]

# kf_retriever_systems = [kid_friend_tf_idf]
# kf_retriever_names = ["TF-IDF"]

kf_queries = pd.read_csv("../data/kid-friend-en/en/inputs/topics.csv")
kf_queries["qid"] = kf_queries["qid"].astype(str)
kf_qrel_file = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
kf_qrel_file["qid"] = kf_qrel_file["qid"].astype(str)
kf_corpus = pd.read_csv("../data/kid-friend-en/en/inputs/documents.csv")

kf_crbp_results = []
for sys, name in zip(kf_retriever_systems, kf_retriever_names):
    score = cRBP(topics=kf_queries, qrels=kf_qrel_file, corpus = kf_corpus, 
                 upper_threshold = 4, lower_threshold = 4, retriever_system=sys, phi=0.8, perquery=False, k=10)
    kf_crbp_results.append([name, score])

kf_crbp_results_df = pd.DataFrame(kf_crbp_results, columns=["name", f"cRBP@10"])
kf_crbp_results_df

,name,cRBP@10
0,TF-IDF,0.065016
1,BM25,0.064960
2,DirichletLM,0.064245
3,MonoT5,0.038889
4,RankVicuna,0.065190
5,RankZephyr,0.039905


In [3]:
kf_rbp_results = kf_rbp_results_df.merge(kf_crbp_results_df, on="name")
kf_rbp_results.to_csv("results/kid-friend/rbp_eval.csv", index=False)
kf_rbp_results

,name,RBP@10,cRBP@10
0,TF-IDF,0.536844,0.065016
1,BM25,0.540305,0.064960
2,DirichletLM,0.527164,0.064245
3,MonoT5,0.300691,0.038889
4,RankVicuna,0.543076,0.065190
5,RankZephyr,0.309961,0.039905
